# Data generation - SimpleWiki - Kaggle

In [ ]:
import json
import os
from pathlib import Path

from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /kaggle/working/Active-Reading--Pattern-Recognition

os.getcwd()

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

CHUNK_SIZE_TOKENS = 4096
CHUNK_OVERLAP_TOKENS = 256

MAX_PARAPHRASE_TOKENS = 4096
MAX_QA_TOKENS = 4096
MAX_STRATEGY_TOKENS = 4096
MAX_ACTIVE_READING_TOKENS = 4096

INPUT_CORPUS = "Datasets/simple_wiki_corpus.json"

BASE_OUT = Path("/kaggle/working/Active-Reading--Pattern-Recognition/generated_simplewiki/")
PARAPHRASE_DIR = BASE_OUT / "paraphrase_outputs"
QA_DIR = BASE_OUT / "synthetic_qa_outputs"
AR_DIR = BASE_OUT / "active_reading_outputs"

for d in [PARAPHRASE_DIR, QA_DIR, AR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map=None,
)

model.to(device)
model.eval()

Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [4]:
@torch.no_grad()
def generate_text(prompt, max_tokens):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    prompt_tokens = inputs.shape[-1]
    prompt_chars = len(prompt)

    outputs = model.generate(
        inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0][prompt_tokens:]
    completion_tokens = generated_tokens.shape[-1]

    completion_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    stats = {
        "prompt_length_toks": int(prompt_tokens),
        "prompt_length_chars": int(prompt_chars),
        "completion_length_toks": int(completion_tokens),
        "completion_length_chars": int(len(completion_text)),
    }

    return completion_text, stats

In [5]:
def chunk_text(tokenizer, text, chunk_size, overlap):
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start = end - overlap
        if start < 0:
            start = 0

    return chunks

In [6]:
with open(INPUT_CORPUS, "r") as f:
    corpus = json.load(f)

print("Loaded documents:", len(corpus))

Loaded documents: 24


In [ ]:
import re

def safe_filename(name: str, max_len: int = 150) -> str:
    name = re.sub(r"[^\w\s.-]", "_", name)
    name = re.sub(r"\s+", "_", name).strip("_")
    return name[:max_len]

## Paraphrasing

In [7]:
PARAPHRASE_PROMPT = """Use the information in the following document to write an informational paragraph in your own words.

Requirements:
- Preserve all factual information, entities, dates, numbers, and relationships.
- Do NOT add new information.
- Do NOT omit important details.
- Output ONLY the paraphrased text.

<document>
{chunk}
</document>
"""

In [8]:
def generate_paraphrase(chunk):
    return generate_text(
        PARAPHRASE_PROMPT.format(chunk=chunk),
        MAX_PARAPHRASE_TOKENS
    )

In [14]:
def get_last_completed_chunk(path):
    """
    Returns the highest chunk_id already written to a JSONL file.
    If file doesn't exist, returns -1.
    """
    if not path.exists():
        return -1

    last_chunk = -1
    with open(path, "r") as f:
        for line in f:
            try:
                obj = json.loads(line)
                last_chunk = max(last_chunk, obj.get("chunk_id", -1))
            except json.JSONDecodeError:
                continue
    return last_chunk

In [ ]:
# Precompute total number of chunks across all documents
total_chunks = 0
doc_chunks = {}

for doc_idx, entry in enumerate(corpus):
    doc_name = safe_filename(entry.get("doc_name", f"doc_{doc_idx}"))
    chunks = chunk_text(
        tokenizer,
        entry["text"],
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS
    )
    doc_chunks[doc_idx] = chunks
    total_chunks += len(chunks)

print(f"Total chunks to process: {total_chunks}")


from tqdm import tqdm

pbar = tqdm(total=total_chunks, desc="Paraphrasing (chunks)")

for doc_idx, entry in enumerate(corpus):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    chunks = doc_chunks[doc_idx]

    out_path = PARAPHRASE_DIR / f"{doc_name}.jsonl"

    # checkpoint
    last_done = get_last_completed_chunk(out_path)
    if last_done >= 0:
        print(f"[resume] {doc_name}: skipping chunks 0–{last_done}")

    with open(out_path, "a") as f:
        for chunk_idx, chunk in enumerate(chunks):

            # skip completed chunks
            if chunk_idx <= last_done:
                pbar.update(1)
                continue

            para, stats = generate_paraphrase(chunk)

            f.write(json.dumps({
                "doc_name": doc_name,
                "chunk_id": chunk_idx,
                "text": para,
                "method": "paraphrase",
                "stats": stats
            }) + "\n")
            f.flush()
            os.fsync(f.fileno())

            pbar.update(1)

pbar.close()

Total chunks to process: 1729


Paraphrasing (chunks):   0%|          | 0/1729 [00:18<?, ?it/s]

[resume] 3M_2018_10K: skipping chunks 0–1



Paraphrasing (chunks):   5%|▍         | 82/1729 [2:14:09<43:35:21, 95.28s/it] 

## Synthetic QA

In [ ]:
SYNTHETIC_QA_PROMPT = """Generate a comprehensive list of fact-based questions and corresponding answers that can be answered explicitly from the document.

Requirements:
- Cover all entities, including people, organizations, dates, locations, quantities, and named concepts.
- Questions must be unambiguous, properly capitalized, and end with a question mark.
- Answers must be as concise as possible and use wording from the document when applicable.
- Output ONE question-answer pair per line.
- Separate the question and answer by a single space.
- Do NOT add any commentary or extra text.
- Do NOT invent information not present in the document.

<document>
{chunk}
</document>
"""

In [ ]:
def generate_synthetic_qa(chunk):
    text, stats = generate_text(
        SYNTHETIC_QA_PROMPT.format(chunk=chunk),
        MAX_QA_TOKENS
    )
    lines = [l.strip() for l in text.split("\n") if l.strip()] 
    return lines, stats

In [ ]:
def parse_qa_lines(lines):
    pairs = []
    used = [False] * len(lines)

    # 1) First, capture two-line pairs: Q? on line i, answer on line i+1
    i = 0
    while i + 1 < len(lines):
        q = lines[i].strip()
        a = lines[i + 1].strip()
        if q.endswith("?") and a and not a.endswith("?"):
            pairs.append((q, a))
            used[i] = used[i + 1] = True
            i += 2
        else:
            i += 1

    # 2) Then, capture one-line pairs: "Question? Answer" (skip lines already used)
    for idx, line in enumerate(lines):
        if used[idx]:
            continue
        line = line.strip()
        if "?" not in line:
            continue

        q_part, a_part = line.split("?", 1)
        q = (q_part.strip() + "?").strip()
        a = a_part.strip()
        if a:
            pairs.append((q, a))

    return pairs

In [ ]:
def get_last_completed_chunk(path):
    if not path.exists():
        return -1

    last_chunk = -1
    with open(path, "r") as f:
        for line in f:
            try:
                obj = json.loads(line)
                last_chunk = max(last_chunk, obj.get("chunk_id", -1))
            except json.JSONDecodeError:
                continue
    return last_chunk

In [ ]:
from tqdm import tqdm
import os

for doc_idx, entry in enumerate(tqdm(corpus, desc="QA documents")):
    doc_name = safe_filename(entry.get("doc_name", f"doc_{doc_idx}"))
    chunks = chunk_text(
        tokenizer,
        entry["text"],
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS
    )
    
    out_path = QA_DIR / f"{doc_name}.jsonl"

    # checkpoint
    last_done = get_last_completed_chunk(out_path)
    if last_done >= 0:
        print(f"[QA resume] {doc_name}: skipping chunks 0–{last_done}")

    with open(out_path, "a") as f:
        for chunk_idx, chunk in enumerate(chunks):

            # skip completed chunks
            if chunk_idx <= last_done:
                continue

            qa_lines, stats = generate_synthetic_qa(chunk)
            qa_pairs = parse_qa_lines(qa_lines)

            for q, a in qa_pairs:
                f.write(json.dumps({
                    "doc_name": doc_name,
                    "chunk_id": chunk_idx,
                    "question": q,
                    "answer": a,
                    "method": "synthetic_qa",
                    "stats": stats
                }) + "\n")

            f.flush()
            os.fsync(f.fileno())

## Active Reading: D.3.1

In [ ]:
D31_STRATEGY_PROMPT = """Consider the following document. 
What are some strategies specific to this document that I can use to help me learn and remember all of the information contained?

Requirements:
- Strategies should be specific to the structure and content of the document.
- Use markdown.
- Prefix each strategy with ##.
- Do NOT summarize the document.
- Do NOT repeat the document content verbatim.

<document>
{chunk}
</document>
"""

In [ ]:
def generate_task_agnostic_strategies(chunk, max_tokens=512):
    prompt = D31_STRATEGY_PROMPT.format(chunk=chunk)
    text, _ = generate_text(prompt, max_tokens)
    return text.strip()

In [ ]:
def split_strategies(strategy_text):
    blocks = strategy_text.split("##")
    return [
        "##" + b.strip()
        for b in blocks
        if b.strip()
    ]

In [ ]:
ACTIVE_READING_PROMPT = """Here is a learning strategy:

{strategy}

Apply this strategy to the following document:

<document>
{chunk}
</document>
"""

In [ ]:
def apply_active_reading(strategy, chunk, max_tokens=1024):
    prompt = ACTIVE_READING_PROMPT.format(
        strategy=strategy,
        chunk=chunk
    )
    return generate_text(prompt, max_tokens)[0]

In [ ]:
def get_last_completed_chunk(path):
    if not path.exists():
        return -1

    last_chunk = -1
    with open(path, "r") as f:
        for line in f:
            try:
                obj = json.loads(line)
                last_chunk = max(last_chunk, obj.get("chunk_id", -1))
            except json.JSONDecodeError:
                continue
    return last_chunk

In [ ]:
AR_DIR = Path("active_reading_outputs")
AR_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from tqdm import tqdm
import os
import random
import json

for doc_idx, entry in enumerate(tqdm(corpus, desc="Active Reading documents")):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    chunks = chunk_text(
        tokenizer,
        entry["text"],
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS
    )

    out_path = AR_DIR / f"{doc_name}.jsonl"

    # checkpoint
    last_done = get_last_completed_chunk(out_path)
    if last_done >= 0:
        print(f"[AR resume] {doc_name}: skipping chunks 0–{last_done}")

    with open(out_path, "a") as f:
        for chunk_idx, chunk in enumerate(chunks):
            
            if chunk_idx % 50 == 0:
                print(f"{doc_name} | chunk {chunk_idx}")
            
            # skip completed chunks
            if chunk_idx <= last_done:
                continue

            # 1. generate strategies (D.3.1)
            strategy_text = generate_task_agnostic_strategies(chunk)

            # 2. split into individual strategies
            strategies = split_strategies(strategy_text)
            if not strategies:
                print(f"[AR warning] {doc_name} chunk {chunk_idx}: no strategies generated, skipping")
                continue

            # optional: limit strategies per chunk
            strategies = random.sample(
                strategies,
                k=min(3, len(strategies))
            )

            # 3. apply each strategy (D.3)
            for strat_idx, strategy in enumerate(strategies):
                ar_out = apply_active_reading(strategy, chunk)

                f.write(json.dumps({
                    "doc_name": doc_name,
                    "chunk_id": chunk_idx,
                    "strategy_id": strat_idx,
                    "strategy_type": "task_agnostic",
                    "strategy": strategy,
                    "active_reading": ar_out,
                    "method": "active_reading_d3"
                }) + "\n")

                f.flush()
                os.fsync(f.fileno())

## Active Reading: D.3.2 (missing for now)